In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -b manuel/cuda https://github.com/JuanCamilo1611/Barnes-Hut.git
%cd Barnes-Hut

name, memory.total [MiB]
Tesla T4, 15360 MiB
Cloning into 'Barnes-Hut'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 190 (delta 63), reused 172 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 277.33 KiB | 7.50 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/Barnes-Hut


In [3]:
!cd cuda && make
!python3 tools/verificar.py --programa "./cuda/sim_cuda" --N 2000

nvcc -O2 -arch=sm_75 -I../c/common -Xcompiler -D_POSIX_C_SOURCE=199309L -o sim_cuda sim_cuda.cu ../c/common/particulas.c ../c/common/directa.c ../c/common/morton.c ../c/common/arbol.c -lm
VERIFICACION
  particulas: 2000
  theta: 0.3  dt: 0.001  pasos: 5  hilos: 1
Ejecutando: ./cuda/sim_cuda /tmp/particulas_r5j6jupi.txt 2000 5 0.001 0.3 0.01 1 /tmp/verificar_aceleraciones.txt
  error maximo:   1.731424 %
  error promedio: 0.119068 %  (2000 particulas)
  derivacion de energia: 0.000000 %

OK: la version pasa la verificacion.


In [4]:
!compute-sanitizer --tool memcheck ./cuda/sim_cuda data/ejemplo_100.txt 100 2 0.001 0.5 0.01 128 /tmp/a.txt | tail -3
!compute-sanitizer --tool racecheck ./cuda/sim_cuda data/ejemplo_100.txt 100 2 0.001 0.5 0.01 128 /tmp/a.txt | tail -3

energia_inicial=-0.911792737173
energia_final=-0.911792724263
========= ERROR SUMMARY: 0 errors
energia_inicial=-0.911792737173
energia_final=-0.911792724263
========= RACECHECK SUMMARY: 0 hazards displayed (0 errors, 0 warnings)


In [5]:
for d in ["uniforme", "concentradas"]:
    !python3 data/generate_particles.py --distribucion {d} --N 200000 --out /tmp/{d}.txt > /dev/null
    print("==", d)
    !./cuda/sim_cuda /tmp/{d}.txt 200000 1 0.001 0.5 0.01 128 /tmp/acc.txt | grep tiempo

== uniforme
tiempo_build=0.096791
tiempo_sort=0.008550
tiempo_walk=0.293736
tiempo_integrate=0.000254
tiempo_total=0.399331
== concentradas
tiempo_build=0.103265
tiempo_sort=0.009175
tiempo_walk=0.498105
tiempo_integrate=0.000282
tiempo_total=0.610828
